# Day 5 模块 2：经营场景深度对比

把表「切成场景」，看营收差在哪。
工具仍是 Day 2 的 `groupby` / 筛选，但问题换成**经营语言**。


In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('day5_cafe_sales.csv'),
    Path('day5') / 'day5_cafe_sales.csv',
    Path('教学课程') / 'day5' / 'day5_cafe_sales.csv',
]
for path in candidate_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('未找到 day5_cafe_sales.csv')

df = pd.read_csv(csv_path)
print(csv_path.resolve())
print('shape:', df.shape)
df.head()


In [ ]:
# 与 Day 3/4 同一套经营特征，方便接模型
weather_score_map = {'晴': 1.0, '多云': 0.8, '阴': 0.6, '小雨': 0.4, '大雨': 0.3}
df = df.copy()
df['weather_score'] = df['weather_label'].map(weather_score_map)

# 经营场景标签（给人看的，不是给模型背的）
df['weekend_label'] = df['is_weekend'].map({0: '工作日', 1: '周末'})
df['promo_label'] = df['promotion'].map({0: '无促销', 1: '有促销'})

print(df[['day', 'weather_label', 'weekend_label', 'promo_label', 'sales']].head())
print('平均营收:', round(df['sales'].mean(), 1))


## 1. 一张总览：均值、中位、最好最差天


In [ ]:
summary = {
    '天数': len(df),
    '平均营收': round(df['sales'].mean(), 1),
    '中位营收': round(df['sales'].median(), 1),
    '最高营收': int(df['sales'].max()),
    '最低营收': int(df['sales'].min()),
    '标准差': round(df['sales'].std(), 1),
}
summary


## 2. 单因子场景：天气 / 周末 / 促销


In [ ]:
print('【天气】平均营收')
print(df.groupby('weather_label')['sales'].agg(['mean', 'count']).round(1).sort_values('mean', ascending=False))
print()
print('【周末】')
print(df.groupby('weekend_label')['sales'].agg(['mean', 'count']).round(1))
print()
print('【促销】')
print(df.groupby('promo_label')['sales'].agg(['mean', 'count']).round(1))


## 3. 交叉场景：周末 × 促销

单看促销可能不够——周末做促销和工作日做促销，感觉往往不同。


In [ ]:
cross = df.pivot_table(
    values='sales',
    index='weekend_label',
    columns='promo_label',
    aggfunc='mean',
).round(1)
cross


## 4. 交叉：天气 × 周末（可选）


In [ ]:
wx = df.pivot_table(
    values='sales',
    index='weather_label',
    columns='weekend_label',
    aggfunc='mean',
).round(1)
wx


## 5. 找出「特别好」和「特别差」的几天


In [ ]:
cols_show = ['day', 'weather_label', 'weekend_label', 'promo_label',
             'price', 'competitors', 'quality', 'sales']
print('营收 TOP 5')
display_top = df.nlargest(5, 'sales')[cols_show]
print(display_top.to_string(index=False))
print()
print('营收 BOTTOM 5')
display_bot = df.nsmallest(5, 'sales')[cols_show]
print(display_bot.to_string(index=False))


## 6. 可选：画一张对比图


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

weather_mean = df.groupby('weather_label')['sales'].mean().sort_values()
plt.figure(figsize=(7, 4))
plt.barh(weather_mean.index.astype(str), weather_mean.values, color='#6f4e37')
plt.xlabel('平均营收')
plt.title('不同天气的平均营收')
plt.tight_layout()
plt.show()


## 7. 经营观察（写数字）

按「发现 → 解释 → 行动」写 2～3 条。例如：

- 发现：有促销日均营收比无促销高约 ____
- 解释：可能吸引了更多到店（只是猜测）
- 行动：可在工作日淡季试点促销，并对比下周数据


In [ ]:
# 在这里写
